# 06 — LIME Point Explanations (Observation-Level)

Produces signed LIME weights at the **individual row level** for a sampled
subset of every run defined in `manifesto_config.pkl`.

| Column | Content |
|---|---|
| Identity columns | `Segment`, `Cust_ID`, `WeekYear`, `UnitSales`, `pred_col` |
| `lime_<feature>` | Signed LIME weight of that feature for this row |

**Sampling**: configurable fraction of rows per Segment (default 10%).  
Full-population LIME is possible but slow — increase `SAMPLE_FRAC` as needed.

**LIME strategy** — identical to `05_lime_explanations.ipynb`:  
- Explainer built once per model from DEV background  
- `predict_fn` built per row for Linear/Neural (Segment fixed); once for Tree  

**Outputs** (one file per run):  
`Outputs/lime_local/{model_key}_lime_{ds_key}.csv`

**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils import load_model_artifacts
from src.lime_utils  import build_lime_explainer, compute_lime_matrix

pd.set_option('display.max_columns', 50)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
LOCAL_DIR    = os.path.join(OUTPUT_DIR, 'lime_local')
os.makedirs(LOCAL_DIR, exist_ok=True)

SAMPLE_FRAC   = 0.10   # 10% per group for point-level explanations
MIN_LIME_ROWS = 20     # floor for very small datasets
ID_COLS       = ['Cust_ID', 'WeekYear', 'Year', 'WeekNum']

def stratified_sample(df, frac, min_rows, group_var=None, seed=42):
    strat_col = group_var if group_var and group_var in df.columns else None
    if strat_col:
        s = df.groupby(strat_col, group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=seed)
        ).reset_index(drop=True)
    else:
        s = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    if len(s) < min_rows:
        s = df.sample(min(min_rows, len(df)), random_state=seed).reset_index(drop=True)
    return s

print('Libraries loaded')

## 1. Load Manifesto

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']
dev_lookup      = {r['model']: r for r in dataset_configs if r['dataset'] == 'DEV'}

print(f'Runs scheduled : {len(dataset_configs)}')

## 2. Point LIME Pipeline — All Runs

In [ ]:
artifacts_cache  = {}   # model_name -> (model_obj, features, aux)
explainer_cache  = {}   # model_name -> (explainer, valid_features)
background_cache = {}   # model_name -> background_df

for run in dataset_configs:
    run_id        = run['run_id']
    model_name    = run['model']
    model_family  = run['model_family']
    model_type    = run['model_type']
    run_gvars     = run['group_vars']
    run_group_var = run_gvars[0] if run_gvars else None

    print(f'\n[{run_id}]  family={model_family}  type={model_type}')

    # ── Load prediction file (from manifest path) ──────────────────────────────
    df = pd.read_csv(run['path'])

    # ── Background ────────────────────────────────────────────────────────────
    if model_name not in background_cache:
        dev_run = dev_lookup.get(model_name)
        background_cache[model_name] = pd.read_csv(dev_run['path']) if dev_run else df
    background_df = background_cache[model_name]

    # ── Load model artifacts + build LIME explainer (cached per model) ─────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts ...')
        model_obj, features, aux = load_model_artifacts(run)
        artifacts_cache[model_name] = (model_obj, features, aux)
        print(f'  Features : {len(features)}')

        print(f'  Building LIME explainer ...')
        explainer, valid_features = build_lime_explainer(background_df, features,
                                                         model_type=model_type)
        explainer_cache[model_name] = (explainer, valid_features)
        print(f'  Valid features for LIME : {len(valid_features)}')
    else:
        model_obj, features, aux = artifacts_cache[model_name]
        explainer, valid_features = explainer_cache[model_name]

    # ── Stratified sample (SAMPLE_FRAC per group_var) ─────────────────────────
    sampled_df = stratified_sample(df, SAMPLE_FRAC, MIN_LIME_ROWS,
                                   group_var=run_group_var)
    print(f'  Sampled rows : {len(sampled_df):,} of {len(df):,}')

    # ── Compute LIME weights ──────────────────────────────────────────────────
    lime_arr = compute_lime_matrix(
        sampled_df, explainer, model_obj, model_family, valid_features, aux,
        model_type=model_type, group_var=run_group_var,
    )
    print(f'  LIME array : {lime_arr.shape}')

    # ── Build wide output ──────────────────────────────────────────────────────
    _gv_list = [run_group_var] if run_group_var else []
    id_available = [c for c in _gv_list + ID_COLS + [run['target_col'], run['pred_col']]
                    if c in sampled_df.columns]
    # deduplicate preserving order
    seen = set(); id_available = [c for c in id_available if not (c in seen or seen.add(c))]
    out = sampled_df[id_available].copy().reset_index(drop=True)

    for j, feat in enumerate(valid_features):
        out[f'lime_{feat}'] = lime_arr[:, j].round(6)

    # ── Save ──────────────────────────────────────────────────────────────────
    model_key = model_name.lower().replace(' ', '_')
    ds_key    = run['dataset'].lower()
    file_path = os.path.join(LOCAL_DIR, f'{model_key}_lime_{ds_key}.csv')
    out.to_csv(file_path, index=False)
    print(f'  Saved: {os.path.basename(file_path)}  ({len(out):,} rows  {len(valid_features)} features)')

print('\nPoint LIME pipeline complete.')

## 3. Quick Sample View

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(LOCAL_DIR, '*.csv')))
if files:
    preview = pd.read_csv(files[0])
    lime_cols = [c for c in preview.columns if c.startswith('lime_')]
    print(f'File : {os.path.basename(files[0])}')
    print(f'Rows : {len(preview):,}  |  LIME features : {len(lime_cols)}')
    display(preview.head(5))